## Important Information

If you are using this for a tutorial then you will need linux to run this code. `openfermion` relies on that. Sorry!

In [ ]:
import perceval as pcvl
import perceval.components.unitary_components as comp
import numpy as np
from scipy.optimize import minimize
import matplotlib.pyplot as plt
from IPython import display
from scipy.linalg import eigh
from scipy.sparse import linalg
from openfermionpyscf import generate_molecular_hamiltonian
from openfermion.chem.molecular_data import MolecularData
from openfermion.transforms.opconversions.conversions import get_fermion_operator
from openfermion.transforms.opconversions.remove_symmetry_qubits import symmetry_conserving_bravyi_kitaev
from openfermion.linalg.sparse_tools import get_sparse_operator
from vqe_perceval import VQESolver
import pyscf    

In [48]:
input_list = list(range(8))
input_str = str(input_list)

In [ ]:
import perceval as pcvl
import perceval.components.unitary_components as comp
import numpy as np
from scipy.optimize import minimize                                     
import pyscf
phi = (1 + np.sqrt(5)) / 2

def build_fermionic_hamiltonian(geometry):
    """
    Construye el Hamiltoniano fermiónico de H2 en base STO-3G via PySCF.
    Retorna el InteractionOperator (openfermion) listo para transformación JW.
    """
    molecule = MolecularData(
        geometry=geometry,
        basis="sto-3g",
        multiplicity=1,
        charge=0,
        description="H2_H7"
    )

    # PySCF resuelve el problema electrónico
    molecule = pyscf.MolecularData(molecule.geometry, molecule.basis, molecule.multiplicity, molecule.charge)
    molecule = pyscf.MolecularData(molecule.geometry, molecule.basis, molecule.multiplicity, molecule.charge)
    molecule = run_pyscf(molecule, run_scf=True, run_fci=True)

    # Hamiltoniano en segunda cuantización
    hamiltonian = molecule.get_molecular_hamiltonian()

    # ── Análisis H7 del estado base ───────────────────────────────────
    n = 1  # H2: 2 electrones → n=1 en Z7
    psi_n   = np.cos(np.pi * phi * n)        # Ψ₁ = O_n_integrity ≈ -0.3624
    par     = np.cos(np.pi * n)              # paridad: -1 → fermionic
    qp      = par * psi_n                    # operador cuasiperiódico completo
    psi_bar = np.cos(np.pi * phi * (7 - n))  # Ψ₆
    delta   = abs(psi_n - psi_bar)           # torsión local

    print(f"{'─'*50}")
    print(f"  H2 Hamiltoniano  ·  STO-3G  ·  H7 prior")
    print(f"{'─'*50}")
    print(f"  E_HF  (SCF)      {molecule.hf_energy:.8f} Ha")
    print(f"  E_FCI (exacta)   {molecule.fci_energy:.8f} Ha")
    print(f"  Correlación      {molecule.fci_energy - molecule.hf_energy:.8f} Ha")
    print(f"{'─'*50}")
    print(f"  n Z₇             {n}")
    print(f"  Ψₙ (O_n)         {psi_n:.8f}   ← theta_0 propuesto")
    print(f"  paridad          {par:+.1f}      ← fermionic")
    print(f"  qp(n)            {qp:.8f}")
    print(f"  δ torsión        {delta:.8f}")
    print(f"{'─'*50}")

    return hamiltonian, molecule

In [ ]:
import pennylane as qml
from pennylane import numpy as np
import math

# ── Constantes H7 ────────────────────────────────────────────────────
phi = (1 + math.sqrt(5)) / 2
O_n = abs(math.cos(math.pi * phi * 1))  # ≈ 0.362375 — punto fijo H7

# ── Molécula ──────────────────────────────────────────────────────────
input = """2
Sample H2 molecule
H 0.3710 0.0 0.0
H -0.3710 0.0 0.0"""

import xyz_parse
molecule = xyz_parse.Molecule.parse(input)

H, qubits = qml.qchem.molecular_hamiltonian(
    molecule.symbols,
    np.array(molecule.coordinates, dtype=np.float64) * 1.88973
)

# ── Circuito ──────────────────────────────────────────────────────────
dev = qml.device("default.qubit", wires=qubits)
electrons = 2
hf = qml.qchem.hf_state(electrons, qubits)

@qml.qnode(dev)
def circuit(param, wires):
    qml.BasisState(hf, wires=wires)
    qml.DoubleExcitation(param, wires=[0, 1, 2, 3])
    return qml.expval(H)

def cost_fn(param):
    return circuit(param, wires=range(qubits))

# ── Inicialización H7 ─────────────────────────────────────────────────
# O_n_integrity como prior en lugar de pi/2
# La paridad de n=1 es -1 (fermionic) → fase negativa
theta = np.array(-O_n)  # -0.362375 en lugar de pi/2 = 1.5708

print(f"H7 init: theta_0 = {theta:.6f}  (O_n_integrity con paridad fermiónica)")
print(f"HF init: theta_0 = {np.pi/2:.6f}  (referencia estándar)")

# ── Optimización ──────────────────────────────────────────────────────
max_iterations = 100
convergence_tolerance = 1e-06
optimizer = qml.GradientDescentOptimizer(stepsize=0.4)

energy_history = []

for i in range(max_iterations):
    theta, prev_energy = optimizer.step_and_cost(cost_fn, theta)
    energy = cost_fn(theta)
    energy_history.append(float(energy))
    if np.abs(energy - prev_energy) <= convergence_tolerance:
        print(f"Convergió en iteración {i+1}")
        break

# ── Análisis H7 del resultado ─────────────────────────────────────────
def h7_classify(energy: float, theta: float) -> dict:
    """Clasifica el estado convergido en el framework H7."""
    # Mapear theta al espacio Z7: theta/pi * 3.5 + 3.5
    n_eff = (theta / math.pi % 1) * 6 + 1  # n efectivo ∈ [1,7)
    n_int = max(1, min(6, round(n_eff)))
    
    par   = math.cos(math.pi * n_int)
    psi_n = math.cos(math.pi * phi * n_int)
    psi_b = math.cos(math.pi * phi * (7 - n_int))
    delta = abs(psi_n - psi_b)
    
    tipo  = "fermionic" if par < 0 else "bosonic"
    
    return {
        "n_efectivo":  n_int,
        "paridad":     par,
        "Ψₙ":          psi_n,
        "Ψ̄ₙ":          psi_b,
        "δ (torsión)": delta,
        "tipo":        tipo,
        "E_ground":    energy,
    }

resultado = h7_classify(float(energy), float(theta))

print(f"\n{'═'*55}")
print(f"  VQE  ·  H₂  ·  Análisis H7")
print(f"{'═'*55}")
for k, v in resultado.items():
    val = f"{v:.8f}" if isinstance(v, float) else str(v)
    print(f"  {k:<18} {val}")
print(f"{'─'*55}")
print(f"  DRIFT canónico Z₇*    0.48796433")
print(f"  δ / DRIFT             {resultado['δ (torsión)'] / 0.48796433:.6f}")
print(f"{'═'*55}")

output = float(energy)

In [ ]:
# The VQESolver implementation is now maintained in vqe_perceval.py.
# This notebook cell is intentionally minimal so the module can be imported cleanly.

# The runtime solver that will be used below is imported from vqe_perceval.py.
